### Robot Electrical Current and Shop Data: Merging and Pre-processing

The electrical current readings (Axis #1 to #3) are given a robot id, an IP address and a shop, then merged with the robot shop table and cleaned up for analysis.

In [4]:
import os
import numpy as np
import pandas as pd
from faker import Faker

## Loading the datasets
The robot shop table first, then the current readings. Only the first three axes and the timestamp are needed.

In [5]:
shops_df = pd.read_csv('robot_shop.csv')
print(shops_df)
print('\nShape of shops_df:', shops_df.shape)

    plant    shopname  TAKT Shop_ID
0  Oshawa    plastics    62  SHP001
1  Oshawa      paints    68  SHP002
2  Oshawa     welding    58  SHP003
3  Oshawa    assembly    60  SHP004
4  Oshawa  final_line    64  SHP005

Shape of shops_df: (5, 4)


In [6]:
current_df = pd.read_csv('robot_data.csv', usecols=['Axis #1', 'Axis #2', 'Axis #3', 'Time'])
print(current_df.head())
print('\nShape of current_df:', current_df.shape)

   Axis #1  Axis #2  Axis #3                      Time
0      0.0      0.0      0.0  2022-10-17T12:18:23.660Z
1      0.0      0.0      0.0  2022-10-17T12:18:25.472Z
2      0.0      0.0      0.0  2022-10-17T12:18:27.348Z
3      0.0      0.0      0.0  2022-10-17T12:18:29.222Z
4      0.0      0.0      0.0  2022-10-17T12:18:31.117Z

Shape of current_df: (39672, 4)


## Initial inspection

In [7]:
shops_df.info()
print()
current_df.info()
print()
print(current_df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   plant     5 non-null      str  
 1   shopname  5 non-null      str  
 2   TAKT      5 non-null      int64
 3   Shop_ID   5 non-null      str  
dtypes: int64(1), str(3)
memory usage: 288.0 bytes

<class 'pandas.DataFrame'>
RangeIndex: 39672 entries, 0 to 39671
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Axis #1  39672 non-null  float64
 1   Axis #2  39672 non-null  float64
 2   Axis #3  39672 non-null  float64
 3   Time     39672 non-null  str    
dtypes: float64(3), str(1)
memory usage: 1.2 MB

            Axis #1       Axis #2       Axis #3
count  39672.000000  39672.000000  39672.000000
mean       0.725743      3.613374      2.710336
std        2.162120      6.879962      5.111901
min        0.000000      0.000000      0.000000
25%        0.000000    

## Creating Robot_id, Ip_address and Shop_ID
Robot_id is unique per reading and Ip_address is a fake private IPv4 from Faker. Shop_ID is drawn from the shop table and capped at 5 shops.

In [8]:
SEED = 42
N_SHOPS = 5

fake = Faker()
Faker.seed(SEED)
rng = np.random.default_rng(SEED)

n = len(current_df)
current_df.insert(0, 'Robot_id', [f'ROB{i:06d}' for i in range(1, n + 1)])
current_df['Ip_address'] = [fake.unique.ipv4_private() for _ in range(n)]

allowed_shops = shops_df['Shop_ID'].sample(N_SHOPS, random_state=SEED).tolist()
current_df['Shop_ID'] = rng.choice(allowed_shops, size=n)

print(current_df.head())
print('\n', current_df['Shop_ID'].value_counts())

    Robot_id  Axis #1  Axis #2  Axis #3                      Time  \
0  ROB000001      0.0      0.0      0.0  2022-10-17T12:18:23.660Z   
1  ROB000002      0.0      0.0      0.0  2022-10-17T12:18:25.472Z   
2  ROB000003      0.0      0.0      0.0  2022-10-17T12:18:27.348Z   
3  ROB000004      0.0      0.0      0.0  2022-10-17T12:18:29.222Z   
4  ROB000005      0.0      0.0      0.0  2022-10-17T12:18:31.117Z   

        Ip_address Shop_ID  
0  192.168.140.208  SHP002  
1     10.52.122.63  SHP001  
2   192.168.44.131  SHP001  
3    192.168.15.65  SHP003  
4     10.13.150.31  SHP003  

 Shop_ID
SHP001    8029
SHP002    7962
SHP005    7929
SHP003    7923
SHP004    7829
Name: count, dtype: int64


## Merging on Shop_ID
Each reading points to one shop, so a left merge on Shop_ID keeps every reading and adds the plant, shop name and TAKT.

In [9]:
merged_df = pd.merge(current_df, shops_df, on='Shop_ID', how='left')
print(merged_df.head())
print('\nShape of the merged DataFrame:', merged_df.shape)

    Robot_id  Axis #1  Axis #2  Axis #3                      Time  \
0  ROB000001      0.0      0.0      0.0  2022-10-17T12:18:23.660Z   
1  ROB000002      0.0      0.0      0.0  2022-10-17T12:18:25.472Z   
2  ROB000003      0.0      0.0      0.0  2022-10-17T12:18:27.348Z   
3  ROB000004      0.0      0.0      0.0  2022-10-17T12:18:29.222Z   
4  ROB000005      0.0      0.0      0.0  2022-10-17T12:18:31.117Z   

        Ip_address Shop_ID   plant  shopname  TAKT  
0  192.168.140.208  SHP002  Oshawa    paints    68  
1     10.52.122.63  SHP001  Oshawa  plastics    62  
2   192.168.44.131  SHP001  Oshawa  plastics    62  
3    192.168.15.65  SHP003  Oshawa   welding    58  
4     10.13.150.31  SHP003  Oshawa   welding    58  

Shape of the merged DataFrame: (39672, 10)


## Saving the merged data

In [10]:
os.makedirs('data', exist_ok=True)
merged_df.to_csv('data/merged_robot_data.csv', index=False)
print('Merged dataframe saved to data/merged_robot_data.csv')

Merged dataframe saved to data/merged_robot_data.csv


## Inspecting the merged dataset

In [11]:
print('Dataset shape:', merged_df.shape)
print('\nColumn names:')
print(merged_df.columns.tolist())
print('\nData types:')
print(merged_df.dtypes)
print('\nMissing values:')
print(merged_df.isnull().sum())
print('\nFirst 5 rows:')
display(merged_df.head())

Dataset shape: (39672, 10)

Column names:
['Robot_id', 'Axis #1', 'Axis #2', 'Axis #3', 'Time', 'Ip_address', 'Shop_ID', 'plant', 'shopname', 'TAKT']

Data types:
Robot_id          str
Axis #1       float64
Axis #2       float64
Axis #3       float64
Time              str
Ip_address        str
Shop_ID           str
plant             str
shopname          str
TAKT            int64
dtype: object

Missing values:
Robot_id      0
Axis #1       0
Axis #2       0
Axis #3       0
Time          0
Ip_address    0
Shop_ID       0
plant         0
shopname      0
TAKT          0
dtype: int64

First 5 rows:


,Robot_id,Axis #1,Axis #2,Axis #3,Time,Ip_address,Shop_ID,plant,shopname,TAKT
0,ROB000001,0.0,0.0,0.0,2022-10-17T12:18:23.660Z,192.168.140.208,SHP002,Oshawa,paints,68
1,ROB000002,0.0,0.0,0.0,2022-10-17T12:18:25.472Z,10.52.122.63,SHP001,Oshawa,plastics,62
2,ROB000003,0.0,0.0,0.0,2022-10-17T12:18:27.348Z,192.168.44.131,SHP001,Oshawa,plastics,62
3,ROB000004,0.0,0.0,0.0,2022-10-17T12:18:29.222Z,192.168.15.65,SHP003,Oshawa,welding,58
4,ROB000005,0.0,0.0,0.0,2022-10-17T12:18:31.117Z,10.13.150.31,SHP003,Oshawa,welding,58


## Handling missing values
The check below covers every column, including the shop fields brought in by the merge.

In [12]:
processed_df = merged_df.copy()

print(processed_df.isnull().sum())

processed_df = processed_df.dropna()
print('\nShape after dropping missing rows:', processed_df.shape)

Robot_id      0
Axis #1       0
Axis #2       0
Axis #3       0
Time          0
Ip_address    0
Shop_ID       0
plant         0
shopname      0
TAKT          0
dtype: int64

Shape after dropping missing rows: (39672, 10)


## Data types
Time is stored as text, so it is converted to a datetime. The axes and TAKT are forced to numeric and the id and shop fields to string.

In [13]:
processed_df['Time'] = pd.to_datetime(processed_df['Time'])

for col in ['Axis #1', 'Axis #2', 'Axis #3', 'TAKT']:
    processed_df[col] = pd.to_numeric(processed_df[col], errors='coerce')

for col in ['Robot_id', 'Ip_address', 'Shop_ID', 'plant', 'shopname']:
    processed_df[col] = processed_df[col].astype(str)

print(processed_df.dtypes)

Robot_id                      str
Axis #1                   float64
Axis #2                   float64
Axis #3                   float64
Time          datetime64[us, UTC]
Ip_address                    str
Shop_ID                       str
plant                         str
shopname                      str
TAKT                        int64
dtype: object


## Duplicates
Robot_id is the primary key, so it should not repeat. The IP addresses are checked as well.

In [14]:
print('Duplicated rows:', processed_df.duplicated().sum())
print('Duplicated Robot_id:', processed_df['Robot_id'].duplicated().sum())
print('Duplicated Ip_address:', processed_df['Ip_address'].duplicated().sum())

Duplicated rows: 0
Duplicated Robot_id: 0
Duplicated Ip_address: 0


In [15]:
processed_df = processed_df.drop_duplicates()
print('Dataset shape after removing exact duplicates:', processed_df.shape)

Dataset shape after removing exact duplicates: (39672, 10)


## Feature engineering
Total current across the three axes, the hour of the reading and an active flag (any axis drawing current).

In [16]:
processed_df['Total Current'] = processed_df[['Axis #1', 'Axis #2', 'Axis #3']].sum(axis=1)
processed_df['Hour'] = processed_df['Time'].dt.hour
processed_df['Active'] = processed_df['Total Current'] > 0

print(processed_df[['Robot_id', 'Time', 'Total Current', 'Hour', 'Active']].head(10))

    Robot_id                             Time  Total Current  Hour  Active
0  ROB000001 2022-10-17 12:18:23.660000+00:00            0.0    12   False
1  ROB000002 2022-10-17 12:18:25.472000+00:00            0.0    12   False
2  ROB000003 2022-10-17 12:18:27.348000+00:00            0.0    12   False
3  ROB000004 2022-10-17 12:18:29.222000+00:00            0.0    12   False
4  ROB000005 2022-10-17 12:18:31.117000+00:00            0.0    12   False
5  ROB000006 2022-10-17 12:18:32.964000+00:00            0.0    12   False
6  ROB000007 2022-10-17 12:18:34.894000+00:00            0.0    12   False
7  ROB000008 2022-10-17 12:18:36.782000+00:00            0.0    12   False
8  ROB000009 2022-10-17 12:18:38.708000+00:00            0.0    12   False
9  ROB000010 2022-10-17 12:18:40.529000+00:00            0.0    12   False


Load levels based on the total current.

In [17]:
def categorize_load(current):
    if current == 0:
        return 'Idle'
    elif current < 15:
        return 'Low'
    elif current < 30:
        return 'Medium'
    else:
        return 'High'

processed_df['Load Level'] = processed_df['Total Current'].apply(categorize_load)

print(processed_df[['Total Current', 'Load Level']].drop_duplicates('Load Level'))

    Total Current Load Level
0         0.00000       Idle
30        7.56068        Low
31       54.13268       High
32       15.15044     Medium


Readings where the current is high on the two heaviest axes at once are flagged.

In [18]:
processed_df['High Load'] = (
    (processed_df['Axis #2'] >= 10) &
    (processed_df['Axis #3'] >= 10)
)

print(processed_df[['Robot_id', 'Axis #2', 'Axis #3', 'High Load']].head(10))

    Robot_id  Axis #2  Axis #3  High Load
0  ROB000001      0.0      0.0      False
1  ROB000002      0.0      0.0      False
2  ROB000003      0.0      0.0      False
3  ROB000004      0.0      0.0      False
4  ROB000005      0.0      0.0      False
5  ROB000006      0.0      0.0      False
6  ROB000007      0.0      0.0      False
7  ROB000008      0.0      0.0      False
8  ROB000009      0.0      0.0      False
9  ROB000010      0.0      0.0      False


## Correlation

In [19]:
correlation = processed_df[
    ['Axis #1', 'Axis #2', 'Axis #3', 'Total Current', 'TAKT']
].corr()

print('Correlation Matrix:')
display(correlation)

Correlation Matrix:


,Axis #1,Axis #2,Axis #3,Total Current,TAKT
Axis #1,1.000000,0.312110,0.370706,0.541077,0.008852
Axis #2,0.312110,1.000000,0.509930,0.886164,-0.002281
Axis #3,0.370706,0.509930,1.000000,0.821649,-0.002223
Total Current,0.541077,0.886164,0.821649,1.000000,-0.000691
TAKT,0.008852,-0.002281,-0.002223,-0.000691,1.000000


## Distributions and shop summary

In [20]:
print('=== Load Level Distribution ===')
print(processed_df['Load Level'].value_counts())

print('\n=== Active Readings ===')
print(processed_df['Active'].value_counts())

print('\n=== High Load Readings ===')
print(processed_df['High Load'].value_counts())

=== Load Level Distribution ===
Load Level
Idle      25713
Medium     6723
Low        4964
High       2272
Name: count, dtype: int64

=== Active Readings ===
Active
False    25713
True     13959
Name: count, dtype: int64

=== High Load Readings ===
High Load
False    38375
True      1297
Name: count, dtype: int64


In [21]:
display(
    processed_df.groupby(['Shop_ID', 'shopname'])[
        ['Axis #1', 'Axis #2', 'Axis #3', 'Total Current']
    ].mean().round(2)
)

,,Axis #1,Axis #2,Axis #3,Total Current
Shop_ID,shopname,,,,
SHP001,plastics,0.73,3.64,2.65,7.02
SHP002,paints,0.75,3.59,2.71,7.05
SHP003,welding,0.71,3.68,2.79,7.18
SHP004,assembly,0.69,3.55,2.67,6.91
SHP005,final_line,0.74,3.61,2.73,7.09


In [22]:
display(processed_df[['Axis #1', 'Axis #2', 'Axis #3', 'Total Current']].describe())

,Axis #1,Axis #2,Axis #3,Total Current
count,39672.000000,39672.000000,39672.000000,39672.000000
mean,0.725743,3.613374,2.710336,7.049453
std,2.162120,6.879962,5.111901,11.466832
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.312710,4.217190,4.586190,13.431515
max,23.609300,51.713230,41.855560,72.174030


## Final dataset

In [23]:
print('Final processed dataset:')
display(processed_df.head(10))

print('\nFinal dataset shape:', processed_df.shape)
print('\nFinal columns:')
print(processed_df.columns.tolist())

Final processed dataset:


,Robot_id,Axis #1,Axis #2,Axis #3,Time,Ip_address,Shop_ID,plant,shopname,TAKT,Total Current,Hour,Active,Load Level,High Load
0,ROB000001,0.0,0.0,0.0,2022-10-17 12:18:23.660000+00:00,192.168.140.208,SHP002,Oshawa,paints,68,0.0,12,False,Idle,False
1,ROB000002,0.0,0.0,0.0,2022-10-17 12:18:25.472000+00:00,10.52.122.63,SHP001,Oshawa,plastics,62,0.0,12,False,Idle,False
2,ROB000003,0.0,0.0,0.0,2022-10-17 12:18:27.348000+00:00,192.168.44.131,SHP001,Oshawa,plastics,62,0.0,12,False,Idle,False
3,ROB000004,0.0,0.0,0.0,2022-10-17 12:18:29.222000+00:00,192.168.15.65,SHP003,Oshawa,welding,58,0.0,12,False,Idle,False
4,ROB000005,0.0,0.0,0.0,2022-10-17 12:18:31.117000+00:00,10.13.150.31,SHP003,Oshawa,welding,58,0.0,12,False,Idle,False
5,ROB000006,0.0,0.0,0.0,2022-10-17 12:18:32.964000+00:00,192.168.214.203,SHP004,Oshawa,assembly,60,0.0,12,False,Idle,False
6,ROB000007,0.0,0.0,0.0,2022-10-17 12:18:34.894000+00:00,10.142.111.3,SHP002,Oshawa,paints,68,0.0,12,False,Idle,False
7,ROB000008,0.0,0.0,0.0,2022-10-17 12:18:36.782000+00:00,10.81.190.221,SHP001,Oshawa,plastics,62,0.0,12,False,Idle,False
8,ROB000009,0.0,0.0,0.0,2022-10-17 12:18:38.708000+00:00,192.168.142.69,SHP005,Oshawa,final_line,64,0.0,12,False,Idle,False
9,ROB000010,0.0,0.0,0.0,2022-10-17 12:18:40.529000+00:00,10.172.86.30,SHP002,Oshawa,paints,68,0.0,12,False,Idle,False



Final dataset shape: (39672, 15)

Final columns:
['Robot_id', 'Axis #1', 'Axis #2', 'Axis #3', 'Time', 'Ip_address', 'Shop_ID', 'plant', 'shopname', 'TAKT', 'Total Current', 'Hour', 'Active', 'Load Level', 'High Load']


In [24]:
os.makedirs('data', exist_ok=True)
processed_df.to_csv('data/processed_robot_data.csv', index=False)

print('Processed dataset saved to: data/processed_robot_data.csv')

Processed dataset saved to: data/processed_robot_data.csv
